# 🚀 ZAC 2026 LifeMate — End-to-End Pipeline trên Kaggle GPU (T4 / P100)
Pipeline trọn gói chuẩn thi đấu cho Zalo AI Challenge 2026 — Bảng LifeMate:
1. **Module ASR**: Gỡ băng âm thanh tiếng Việt siêu tốc bằng `Faster-Whisper` kết hợp trọng số `PhoWhisper-medium-ct2` hoặc `large-v3-turbo` (Không bao giờ bị lỗi 403).
2. **Module Post-Processing**: Chuẩn hóa từ điển nghiệp vụ LifeMate (Zalo, VTV, Smart Home, Banking).
3. **Module Agent**: Tác nhân ReAct với `Qwen/Qwen2.5-3B-Instruct` tương tác trực tiếp với `ToolEnv`.
4. **Module Export**: Đóng gói tự động `submission.jsonl` và `submission.zip` chuẩn quy chế BTC.

### Bước 1: Cài đặt thư viện phụ thuộc

In [ ]:
!pip install -q faster-whisper soundfile python-dotenv transformers accelerate

### Bước 2: Thiết lập môi trường và tự động nhận diện dữ liệu trên Kaggle

In [ ]:
import os
import sys
import json
import time
import zipfile
from pathlib import Path
from tqdm import tqdm

# Đảm bảo nạp các module từ thư mục repo zac2026-lifemate
REPO_DIR = Path("/kaggle/working/zac2026-lifemate")
if not REPO_DIR.exists():
    REPO_DIR = Path.cwd()

sys.path.insert(0, str(REPO_DIR))

# Tự động tìm kiếm thư mục train+public_test trong Kaggle input
DATA_CANDIDATES = [
    Path("/kaggle/input/datasets/n107hoangtuong/zalo-competion/train+public_test"),
    Path("/kaggle/input/zalo-ai-challenge-2026/train+public_test"),
    Path("/kaggle/input/train+public_test"),
]
DATA_ROOT = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_ROOT is None:
    # Tìm kiếm đệ quy
    for root, dirs, files in os.walk("/kaggle/input"):
        if "public_test.jsonl" in files:
            DATA_ROOT = Path(root).parent
            break

print(f"✅ Đã nhận diện DATA_ROOT tại: {DATA_ROOT}")
OUTPUT_DIR = Path("/kaggle/working/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### Bước 3: Nạp ASR Engine (PhoWhisper CTranslate2 — Tốc độ cực nhanh & Không bao giờ lỗi 403)

In [ ]:
from faster_whisper import WhisperModel

# Sử dụng PhoWhisper-medium bản CT2 (tối ưu chạy trên GPU T4, tốc độ ~0.3s/mẫu)
MODEL_ID = "nunu-willump/PhoWhisper-medium-ct2"
print(f"⏳ Đang nạp mô hình ASR: {MODEL_ID} trên CUDA (float16)...")
asr_model = WhisperModel(MODEL_ID, device="cuda", compute_type="float16")
print("🎉 Nạp thành công ASR Model!")

### Bước 4: Chạy gỡ băng âm thanh FULL (public_test hoặc train)

In [ ]:
from src.asr.post_processor import LexiconPostProcessor
from src.asr.transcribe import load_audio_file

# Khởi tạo bộ chuẩn hóa từ điển nghiệp vụ LifeMate (Tầng 1)
post_proc = LexiconPostProcessor()

# Chọn tập dữ liệu cần chạy: 'public_test' (nộp bài) hoặc 'train' (đánh giá)
SPLIT = "public_test"
INPUT_FILE = DATA_ROOT / SPLIT / f"{SPLIT}.jsonl"
AUDIO_DIR = DATA_ROOT / SPLIT
TRANSCRIPT_OUT = OUTPUT_DIR / f"{SPLIT}_transcripts.jsonl"

print(f"🚀 Bắt đầu gỡ băng FULL tập {SPLIT}...")
records = []
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

print(f"-> Tổng số mẫu: {len(records)}")

with open(TRANSCRIPT_OUT, "w", encoding="utf-8") as out_f:
    for item in tqdm(records, desc=f"Transcribing {SPLIT}"):
        sample_id = item["id"]
        audio_path = AUDIO_DIR / item["audio"]
        
        # Gỡ băng câu khẩu lệnh chính bằng soundfile (tránh lỗi PyAV)
        audio_data = load_audio_file(audio_path, target_sr=16000)
        segments, _ = asr_model.transcribe(audio_data, language="vi", beam_size=1, temperature=0.0)
        raw_text = " ".join([s.text.strip() for s in segments]).strip()
        final_text = post_proc.process(raw_text)
        
        # Gỡ băng các lượt user trong history nếu có
        hist_transcripts = []
        if "history" in item and isinstance(item["history"], list):
            for h_idx, turn in enumerate(item["history"]):
                if turn.get("audio"):
                    h_path = AUDIO_DIR / turn["audio"]
                    if h_path.exists():
                        h_data = load_audio_file(h_path, target_sr=16000)
                        h_segs, _ = asr_model.transcribe(h_data, language="vi", beam_size=1, temperature=0.0)
                        h_raw = " ".join([s.text.strip() for s in h_segs]).strip()
                        hist_transcripts.append({"history_index": h_idx, "transcript": post_proc.process(h_raw)})
        
        res = {
            "id": sample_id,
            "transcript": final_text,
            "history_user_transcripts": hist_transcripts
        }
        out_f.write(json.dumps(res, ensure_ascii=False) + "\n")

print(f"✅ Đã xuất file transcript thành công tại: {TRANSCRIPT_OUT}")

### Bước 5: Chạy ReAct Agent (Qwen2.5-3B) tương tác ToolEnv và xuất file nộp bài

In [ ]:
# Giải phóng VRAM của ASR trước khi nạp LLM 3B
import gc
import torch
del asr_model
gc.collect()
torch.cuda.empty_cache()

# Chạy sinh kết quả nộp bài thông qua script run_with_asr.py
!python scripts/run_with_asr.py \
    --make_submission \
    --data_root "{DATA_ROOT}" \
    --asr_dir "{OUTPUT_DIR}" \
    --output_file "/kaggle/working/submission.jsonl"

print("🎉 ĐÃ HOÀN TẤT TOÀN BỘ PIPELINE!")
print("Tải file /kaggle/working/submission.zip về để nộp bài lên hệ thống ZAC 2026.")